# /dedup-check · /dedup-check-embed · /dedup/bootstrap — Endpoint Evaluation

For each fixture group:
1. Uses `/dedup/bootstrap` to seed the index with the first article.
2. Submits the second (and any further) articles via `/dedup-check`.
3. Verifies that expected duplicate pairs are detected at the correct stage.

The service state accumulates across groups within a single run — **restart the service between runs** to get a clean index.

**Prerequisite:** NLP service running with an **empty** dedup index. `/readyz` → 200.

In [1]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('dedup_cases.json')
print(f'Loaded {len(cases)} dedup groups')

Loaded 8 dedup groups


In [2]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [3]:
results = []

for group in cases:
    articles = group['articles']
    expected_pairs = {tuple(sorted(p)) for p in group.get('expected_pairs', [])}
    expected_stage = group.get('expected_stage')  # may be None for no-dup groups

    print(f"── {group['id']}: {group['description']}")

    # Bootstrap with the first article to seed the index
    seed = articles[0]
    boot_resp = requests.post(
        f'{NLP_BASE_URL}/dedup/bootstrap',
        json={'articles': [{'article_id': seed['article_id'], 'text': seed['text']}]},
        headers=HEADERS,
    )
    assert boot_resp.status_code == 200, \
        f"bootstrap failed: {boot_resp.status_code} {boot_resp.text}"

    # Check each subsequent article
    detected_pairs = set()
    stage_ok = True
    for article in articles[1:]:
        t0 = time.monotonic()
        resp = requests.post(
            f'{NLP_BASE_URL}/dedup-check',
            json={'article_id': article['article_id'], 'text': article['text']},
            headers=HEADERS,
        )
        latency = time.monotonic() - t0
        assert resp.status_code == 200, \
            f"{article['article_id']}: HTTP {resp.status_code} — {resp.text}"
        data = resp.json()

        print(f"   {article['article_id']}  duplicate_of={data['duplicate_of']!r}  "
              f"stage={data['stage']!r}  score={data['score']}  indexed={data['indexed']}  "
              f"{latency:.2f}s")

        if data['duplicate_of'] is not None:
            pair = tuple(sorted([article['article_id'], data['duplicate_of']]))
            detected_pairs.add(pair)
            if expected_stage and data['stage'] != expected_stage:
                stage_ok = False

    pairs_ok = detected_pairs == expected_pairs
    passed   = pairs_ok and stage_ok
    icon     = '✅' if passed else '❌'
    print(f"   {icon} pairs_ok={pairs_ok}  stage_ok={stage_ok}")
    if not pairs_ok:
        print(f"      expected: {expected_pairs}")
        print(f"      got:      {detected_pairs}")
    print()

    results.append({
        'id':         group['id'],
        'pairs_ok':   pairs_ok,
        'stage_ok':   stage_ok,
        'pass':       passed,
    })

── dedup-001: Exact duplicate
   dedup-001-b  duplicate_of='dedup-001-a'  stage='minhash'  score=1.0  indexed=False  0.10s
   ✅ pairs_ok=True  stage_ok=True

── dedup-002: Near-duplicate: different headline/byline, same body
   dedup-002-b  duplicate_of=None  stage=None  score=None  indexed=True  0.10s
   ❌ pairs_ok=False  stage_ok=True
      expected: {('dedup-002-a', 'dedup-002-b')}
      got:      set()

── dedup-003: Paraphrase / rewrite — same facts, different wording
   dedup-003-b  duplicate_of=None  stage=None  score=None  indexed=True  0.11s
   ❌ pairs_ok=False  stage_ok=True
      expected: {('dedup-003-a', 'dedup-003-b')}
      got:      set()

── dedup-004: Clearly distinct articles — same topic, different events
   dedup-004-b  duplicate_of=None  stage=None  score=None  indexed=True  0.11s
   ✅ pairs_ok=True  stage_ok=True

── dedup-005: Same article seen again after indexing (idempotency)
   dedup-005-a  duplicate_of='dedup-005-a'  stage='minhash'  score=1.0  indexed=Fals

In [4]:
passing    = [r for r in results if r['pass']]
pairs_pass = sum(1 for r in results if r['pairs_ok'])
stage_pass = sum(1 for r in results if r['stage_ok'])

print_scorecard('/dedup-check', {
    'Groups':                  len(results),
    'Passing (all checks)':    f'{len(passing)}/{len(results)}',
    'Correct pair detection':  f'{pairs_pass}/{len(results)}',
    'Correct stage':           f'{stage_pass}/{len(results)}',
})


  /dedup-check
  Groups                              8
  Passing (all checks)                4/8
  Correct pair detection              4/8
  Correct stage                       8/8



## /dedup-check-embed — smoke test

Verifies the pre-computed embedding path by calling `/extract` then `/dedup-check-embed`.

In [5]:
from _scorecard import load_fixture
sum_cases = load_fixture('summarize_cases.json')
article = sum_cases[0]

# Get embedding from /extract
ext = requests.post(
    f'{NLP_BASE_URL}/extract',
    json={'article_id': article['article_id'], 'text': article['text']},
    headers=HEADERS,
)
assert ext.status_code == 200, f"/extract failed: {ext.text}"
embedding = ext.json()['embedding_raw']

# Check via /dedup-check-embed
resp = requests.post(
    f'{NLP_BASE_URL}/dedup-check-embed',
    json={'article_id': article['article_id'], 'embedding_raw': embedding},
    headers=HEADERS,
)
assert resp.status_code == 200, f"/dedup-check-embed failed: {resp.text}"
data = resp.json()
print(f"/dedup-check-embed smoke test ✅")
print(f"  article_id={data['article_id']}  duplicate_of={data['duplicate_of']!r}  "
      f"stage={data['stage']!r}  indexed={data['indexed']}")

/dedup-check-embed smoke test ✅
  article_id=sum-001  duplicate_of=None  stage=None  indexed=True
